# 🔍 Multimodal Fake News Detection — Google Colab Training
**BERT + ViT + Cross-Modal Attention** framework for detecting fake news.

This notebook lets you train the full model on **Colab's free GPU**.

In [ ]:
# ============================================================
# STEP 1: GPU Check & Setup
# ============================================================
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA:', torch.cuda.is_available())
PROJECT_DIR = '/content/fake_news_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
print('Project dir:', PROJECT_DIR)

In [ ]:
# ============================================================
# STEP 2: Install Dependencies
# ============================================================
!pip install -q transformers datasets scikit-learn seaborn tqdm tensorboard pyyaml

In [ ]:
# ============================================================
# STEP 3: Upload your project zip OR clone from GitHub
# ============================================================
# OPTION A — Upload zip (run this cell)
from google.colab import files
import zipfile, shutil
print('Upload your project zip file...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/_tmp')
# Find the root that contains 'src'
import glob
roots = glob.glob('/content/_tmp/**/src', recursive=True)
if roots:
    src_parent = os.path.dirname(roots[0])
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(src_parent, PROJECT_DIR)
    print(f'Extracted to {PROJECT_DIR}')
else:
    print('ERROR: Could not find src/ in zip. Check your zip structure.')
shutil.rmtree('/content/_tmp', ignore_errors=True)

In [ ]:
# OPTION B — Clone from GitHub (uncomment & edit)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/fake_news_project

In [ ]:
# ============================================================
# STEP 4: Upload Dataset
# ============================================================
# Upload ISOT dataset (Fake.csv + True.csv) or your CSV with text+label columns
DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'isot')
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Upload your dataset CSV files to: {DATA_DIR}')
print('For ISOT: upload Fake.csv and True.csv')
print('For custom: upload train.csv / test.csv with text and label columns')
from google.colab import files as gfiles
up = gfiles.upload()
for name, data in up.items():
    dest = os.path.join(DATA_DIR, name)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest} ({len(data)/1e6:.1f} MB)')
print('Done!')

In [ ]:
# ============================================================
# STEP 5: Verify Project Structure
# ============================================================
import sys
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# Show structure
for root, dirs, fls in os.walk(PROJECT_DIR):
    dirs[:] = [d for d in dirs if d not in {'__pycache__', '.git', '.venv', '.venv310', 'node_modules'}]
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    sub = '  ' * (level + 1)
    for f in fls[:10]:
        print(f'{sub}{f}')

In [ ]:
# ============================================================
# STEP 6: Load & Adjust Config for Colab
# ============================================================
import yaml
config_path = os.path.join(PROJECT_DIR, 'config', 'config.yaml')
with open(config_path) as f:
    config = yaml.safe_load(f)

# --- Colab-optimized overrides ---
config['training']['batch_size'] = 8        # Lower for Colab GPU RAM
config['training']['num_epochs'] = 10       # Adjust as needed
config['training']['fp16'] = True
config['data']['num_workers'] = 2           # Colab has limited CPUs
config['data']['data_dir'] = DATA_DIR
# Uncomment next line to use generic CSV adapter instead of ISOT:
# config['data']['dataset_name'] = 'generic'

# Save adjusted config
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('=== Colab Training Config ===')
print(f"  Dataset    : {config['data']['dataset_name']}")
print(f"  Data dir   : {config['data']['data_dir']}")
print(f"  Batch size : {config['training']['batch_size']}")
print(f"  Epochs     : {config['training']['num_epochs']}")
print(f"  LR         : {config['training']['learning_rate']}")
print(f"  FP16       : {config['training']['fp16']}")

In [ ]:
# ============================================================
# STEP 7: Train the Model 🚀
# ============================================================
import random, numpy as np, torch
from src.models import MultimodalFakeNewsDetector
from src.data.dataset import get_dataloader
from src.training.trainer import Trainer

# Seed
seed = config['training'].get('seed', 42)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# DataLoaders
dataloaders = get_dataloader(
    data_dir=config['data']['data_dir'],
    dataset_name=config['data']['dataset_name'],
    tokenizer_name=config['model']['text_encoder']['name'],
    max_length=config['model']['text_encoder']['max_length'],
    image_size=config['model']['image_encoder']['image_size'],
    batch_size=config['training']['batch_size'],
    val_split=config['training'].get('val_split', 0.15),
    test_split=config['training'].get('test_split', 0.15),
    num_workers=config['data'].get('num_workers', 2),
    pin_memory=True,
    seed=seed,
)

# Model
model = MultimodalFakeNewsDetector(config)
params = model.get_trainable_params()
print(f"Params: {params['trainable']:,} trainable / {params['total']:,} total ({params['trainable_pct']:.1f}%)")

# Train
trainer = Trainer(model=model, config=config, device=device)
MODE = 'multimodal'  # Change to 'text_only' or 'image_only' for ablation
result = trainer.train(
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    mode=MODE,
)
print(f"\nBest epoch: {result['best_epoch']}  |  Best F1: {result['best_val_f1']:.4f}")

In [ ]:
# ============================================================
# STEP 8: Evaluate on Test Set
# ============================================================
print('Running final evaluation on test set...')
test_metrics = trainer.evaluate(
    test_loader=dataloaders['test'],
    mode=MODE,
    generate_plots=True,
)

# Show plots inline
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
from pathlib import Path
plots_dir = Path(config.get('logging',{}).get('log_dir','./logs')) / 'metrics'
for img in sorted(plots_dir.glob('*.png')):
    print(f'\n--- {img.name} ---')
    display(IPImage(filename=str(img)))

In [ ]:
# ============================================================
# STEP 9: Plot Training History
# ============================================================
import matplotlib.pyplot as plt

history = result['history']
epochs = [h['epoch'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs, [h['train_loss'] for h in history], 'b-o', label='Train')
axes[0].plot(epochs, [h['val_loss'] for h in history], 'r-o', label='Val')
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, [h['train_accuracy'] for h in history], 'b-o', label='Train')
axes[1].plot(epochs, [h['val_accuracy'] for h in history], 'r-o', label='Val')
axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

# F1
axes[2].plot(epochs, [h['train_f1'] for h in history], 'b-o', label='Train')
axes[2].plot(epochs, [h['val_f1'] for h in history], 'r-o', label='Val')
axes[2].set_title('F1 Score', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 10: Download Trained Model
# ============================================================
from google.colab import files
import shutil

# Zip checkpoints + results
ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints')
results_dir = os.path.join(PROJECT_DIR, 'results')
archive = '/content/trained_model'
dirs_to_zip = [d for d in [ckpt_dir, results_dir] if os.path.exists(d)]

os.makedirs('/content/export', exist_ok=True)
for d in dirs_to_zip:
    dest = os.path.join('/content/export', os.path.basename(d))
    if os.path.exists(dest): shutil.rmtree(dest)
    shutil.copytree(d, dest)

shutil.make_archive(archive, 'zip', '/content/export')
print('Downloading trained_model.zip...')
files.download(archive + '.zip')

In [ ]:
# ============================================================
# STEP 11 (Optional): Ablation Study
# ============================================================
# Uncomment to compare multimodal vs text-only vs image-only

# from src.training.metrics import compare_models
# results_all = {}
# for mode in ['multimodal', 'text_only', 'image_only']:
#     print(f'\n=== {mode.upper()} ===')
#     m = trainer.evaluate(dataloaders['test'], mode=mode, generate_plots=False)
#     results_all[mode] = m
#     print(f"  Acc: {m['accuracy']:.4f}  F1: {m['f1']:.4f}")
# compare_models(results_all, save_dir='./results')